# Agents SDK tools and function calling

## Learning goals

- Turn typed Python functions into model-visible tools
- Inspect generated names, descriptions, and JSON schemas
- Inject trusted application context without exposing it as model arguments
- Test a function tool offline through its SDK invocation boundary
- Understand errors, conditional availability, timeouts, and approvals


## What `@function_tool` does

The decorator inspects the function signature and docstring, creates a JSON schema for model-generated arguments, validates arguments, and adapts sync or async Python execution into the SDK tool loop.

It does not decide authorization, make side effects safe, or turn a docstring into an enforceable policy. The decorated function remains application code and must check trusted context before accessing data or performing actions.


## Mental model

```mermaid
sequenceDiagram
  participant M as Model
  participant S as Agents SDK
  participant A as App authorization
  participant F as Python function
  M-->>S: Tool name plus JSON arguments
  S->>S: Match registered tool and validate schema
  S->>A: Check context, enablement, approval, guardrails
  alt permitted
    S->>F: Invoke typed arguments
    F-->>S: Result or controlled error
    S-->>M: Tool output item
  else denied
    S-->>M: Rejection, interruption, or tripwire
  end
```

The model sees the public schema. Trusted application context—user ID, tenant, permissions, database handles—travels separately through `RunContextWrapper.context`.


## Start with narrow typed tools

Type annotations define argument types. The function docstring becomes the tool description, and documented arguments become property descriptions. Prefer small functions with names that describe one capability.


In [ ]:
from typing import Literal

from agents import function_tool


@function_tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers.

    Args:
        a: First numeric factor.
        b: Second numeric factor.
    """
    result = a * b
    if abs(result) > 1e12:
        raise ValueError("Result exceeds the teaching limit.")
    return result


print("Tool name:", multiply.name)
print("Description:", multiply.description)
print("Strict schema:", multiply.strict_json_schema)
print("Schema:", multiply.params_json_schema)


## Use context for trusted application state

If the first parameter is `RunContextWrapper[...]`, the SDK injects it locally and omits it from the model-facing argument schema. The tool can then authorize a requested city against trusted session state rather than accepting permissions from the model.


In [ ]:
from dataclasses import dataclass

from agents import RunContextWrapper


@dataclass(frozen=True)
class TravelContext:
    user_id: str
    allowed_cities: frozenset[str]
    plan: Literal["free", "pro"] = "free"


@function_tool
def get_weather(
    ctx: RunContextWrapper[TravelContext],
    city: str,
    units: Literal["celsius", "fahrenheit"],
) -> dict[str, str]:
    """Return deterministic offline weather for an authorized city.

    Args:
        city: City to look up.
        units: Temperature unit to use.
    """
    if city.lower() not in ctx.context.allowed_cities:
        raise PermissionError("City is not authorized for this session.")
    temperature = "30 C" if units == "celsius" else "86 F"
    return {
        "city": city,
        "temperature": temperature,
        "condition": "offline mock",
    }


print(get_weather.params_json_schema)
assert "ctx" not in get_weather.params_json_schema["properties"]


## Test the SDK tool boundary offline

Calling `on_invoke_tool` exercises the generated JSON parsing and context-aware wrapper without a model request. This is a low-level SDK test surface; most application code should let `Runner` invoke tools during a run.


In [ ]:
from agents.tool_context import ToolContext


travel_context = TravelContext(
    user_id="demo-user",
    allowed_cities=frozenset({"jaipur", "jodhpur"}),
)
tool_input = '{"city": "Jaipur", "units": "celsius"}'
tool_context = ToolContext(
    context=travel_context,
    tool_name=get_weather.name,
    tool_call_id="offline-call-1",
    tool_arguments=tool_input,
)

weather_result = await get_weather.on_invoke_tool(tool_context, tool_input)
print(weather_result)


## Conditional availability and approval

`is_enabled` controls whether a tool is offered at runtime; use it for plan, tenant, or feature availability. `needs_approval` pauses before execution so the application can collect approval. Neither option should rely on untrusted model text for identity. In the current SDK, decorator-level `timeout` is supported for async function-tool handlers, which is why the premium lookup below is async.


In [ ]:
def pro_tool_enabled(ctx: RunContextWrapper[TravelContext], agent) -> bool:
    return ctx.context.plan == "pro"


@function_tool(is_enabled=pro_tool_enabled, timeout=2.0)
async def search_premium_guides(city: str) -> str:
    """Search premium local guides for a city."""
    return f"Offline premium-guide result for {city}."


@function_tool(needs_approval=True)
def cancel_booking(booking_reference: str) -> str:
    """Cancel a booking after explicit application approval."""
    return f"Teaching stub: did not cancel {booking_reference}."


print("Premium tool enabled by function:", callable(search_premium_guides.is_enabled))
print("Cancellation requires approval:", cancel_booking.needs_approval)


## Attach tools to an agent

Tool descriptions should help selection, while instructions explain when tools are appropriate. Do not tell the agent it has a tool that is conditionally disabled for the current run; the SDK's runtime inventory should remain the source of truth.


In [ ]:
import os

from agents import Agent, ModelSettings, Runner

tool_agent = Agent[TravelContext](
    name="Tool-aware travel helper",
    instructions=(
        "Use tools only when they provide information needed for the answer. "
        "Never claim a side effect completed unless the tool result confirms it."
    ),
    model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
    model_settings=ModelSettings(max_tokens=300, store=False),
    tools=[multiply, get_weather, search_premium_guides, cancel_booking],
)

print([tool.name for tool in tool_agent.tools])


In [ ]:
RUN_LIVE_CALL = False

if RUN_LIVE_CALL:
    result = await Runner.run(
        tool_agent,
        "What is the mock weather in Jaipur in celsius?",
        context=travel_context,
        max_turns=5,
    )
    print(result.final_output)
else:
    print("Live tool-calling run skipped.")


## SDK tools versus the manual registry

| Concern | Manual runtime | Agents SDK |
|---|---|---|
| Schema generation | Write Pydantic/JSON schema yourself | Derived from type hints/docstrings |
| Argument parsing | Dispatcher code | SDK function schema adapter |
| Tool loop | Your loop | `Runner` |
| Authorization | Your code | Still your code/context/guardrails |
| Approval | Build interruption state | `needs_approval` plus resumable run state |
| Timeouts/errors | Build wrappers | Tool options plus application policy |
| Observability | Custom events | SDK tool items/spans plus app logs |

The SDK removes protocol plumbing; it does not remove product-specific policy.


## Failure cases and improvements

- **Vague docstring:** describe when the tool helps and what each argument means.
- **Broad signature:** expose narrow validated arguments rather than a generic dictionary.
- **Permission encoded in prompt:** enforce with authenticated context and runtime checks.
- **Raw exception sent to user:** configure a safe error function and log protected diagnostics.
- **Write tool executes immediately:** require approval and idempotency.
- **Tool always offered:** use `is_enabled` to reduce irrelevant or unauthorized capabilities.
- **Slow tool blocks a run:** set a timeout and define retry/idempotency behavior.


## Exercise

1. Invoke `multiply` offline with valid and invalid JSON.
2. Try an unauthorized city and inspect the safe tool result.
3. Add a context-aware `search_notes` tool that scopes data by user ID.
4. Define what approval record a real `cancel_booking` tool would require.
5. Compare the generated schemas with the registry schemas from the first-principles notebook.


## Official references

- [Agents SDK tools](https://openai.github.io/openai-agents-python/tools/)
- [Function tool API](https://openai.github.io/openai-agents-python/ref/tool/)
- [Tool context](https://openai.github.io/openai-agents-python/ref/tool_context/)


## Summary

`@function_tool` converts a typed function and docstring into a strict model-visible schema and SDK invocation wrapper. Keep trusted state in context, test the invocation boundary offline, conditionally expose capabilities, require approval for side effects, and remember that application authorization remains yours.
